In [ ]:
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

In [2]:
import sys
sys.path.append('src')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from morocco_datasets.morocco_dataset import MoroccoWeatherDataset
from models.conv_lstm_baseline import ConvLSTMBaseline
from models.vit_nowcasting import ViTNowcasting
import random

OSError: [WinError 1114] Une routine d’initialisation d’une bibliothèque de liens dynamiques (DLL) a échoué. Error loading "C:\Users\Admin\AppData\Roaming\Python\Python311\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# Charger les datasets de test
data_dir = 'dataset'
sequence_length = 3
forecast_horizon = 1
test_dataset = MoroccoWeatherDataset(data_dir, sequence_length, forecast_horizon, 'test')
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
# Charger les modèles
baseline_model = ConvLSTMBaseline(input_channels=4, hidden_dim=16, kernel_size=(3,3), num_layers=1)
baseline_model.load_state_dict(torch.load('experiments/baseline_run_001/model.pt', map_location='cpu'))
baseline_model.eval()

vit_model = ViTNowcasting(input_channels=4, patch_size=16, embed_dim=384, depth=6, num_heads=6, mlp_ratio=4.0, seq_length=3, forecast_horizon=1)
vit_model.load_state_dict(torch.load('experiments/vit_run_001/model.pt', map_location='cpu'))
vit_model.eval()

print("Models loaded")

In [ ]:
# Fonction pour calculer RMSE et MAE
def compute_metrics(pred, target):
    pred = pred.detach().numpy()
    target = target.detach().numpy()
    rmse = np.sqrt(np.mean((pred - target)**2))
    mae = np.mean(np.abs(pred - target))
    return rmse, mae

# Évaluer sur le test set
baseline_rmse = []
baseline_mae = []
vit_rmse = []
vit_mae = []

with torch.no_grad():
    for X, y in test_loader:
        # Baseline
        baseline_pred = baseline_model(X)
        rmse, mae = compute_metrics(baseline_pred, y)
        baseline_rmse.append(rmse)
        baseline_mae.append(mae)
        
        # ViT
        vit_pred = vit_model(X)
        rmse, mae = compute_metrics(vit_pred, y)
        vit_rmse.append(rmse)
        vit_mae.append(mae)

baseline_rmse_avg = np.mean(baseline_rmse)
baseline_mae_avg = np.mean(baseline_mae)
vit_rmse_avg = np.mean(vit_rmse)
vit_mae_avg = np.mean(vit_mae)

print(f"Baseline - RMSE: {baseline_rmse_avg:.4f}, MAE: {baseline_mae_avg:.4f}")
print(f"ViT - RMSE: {vit_rmse_avg:.4f}, MAE: {vit_mae_avg:.4f}")

In [ ]:
# Visualisations pour 3 exemples aléatoires
random_indices = random.sample(range(len(test_dataset)), 3)

fig, axes = plt.subplots(3, 6, figsize=(18, 9))

for i, idx in enumerate(random_indices):
    X, y = test_dataset[idx]
    X = X.unsqueeze(0)
    y = y.unsqueeze(0)
    
    with torch.no_grad():
        baseline_pred = baseline_model(X)
        vit_pred = vit_model(X)
    
    # Afficher la vraie valeur (canal 0, par exemple satellite)
    axes[i, 0].imshow(y[0, 0, 0].numpy(), cmap='viridis')
    axes[i, 0].set_title('Ground Truth')
    axes[i, 0].axis('off')
    
    # Prédiction Baseline
    axes[i, 1].imshow(baseline_pred[0, 0, 0].numpy(), cmap='viridis')
    axes[i, 1].set_title('Baseline Pred')
    axes[i, 1].axis('off')
    
    # Erreur Baseline
    error = np.abs(baseline_pred[0, 0, 0].numpy() - y[0, 0, 0].numpy())
    axes[i, 2].imshow(error, cmap='hot')
    axes[i, 2].set_title('Baseline Error')
    axes[i, 2].axis('off')
    
    # Prédiction ViT
    axes[i, 3].imshow(vit_pred[0, 0, 0].numpy(), cmap='viridis')
    axes[i, 3].set_title('ViT Pred')
    axes[i, 3].axis('off')
    
    # Erreur ViT
    error = np.abs(vit_pred[0, 0, 0].numpy() - y[0, 0, 0].numpy())
    axes[i, 4].imshow(error, cmap='hot')
    axes[i, 4].set_title('ViT Error')
    axes[i, 4].axis('off')
    
    # Différence
    diff = vit_pred[0, 0, 0].numpy() - baseline_pred[0, 0, 0].numpy()
    axes[i, 5].imshow(diff, cmap='bwr')
    axes[i, 5].set_title('ViT - Baseline')
    axes[i, 5].axis('off')

plt.tight_layout()
plt.show()